# Generic - Rust

All 21 Rust examples from [docs/generic.md](https://platob.github.io/yggdryl/generic/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Shared vocabulary

In [ ]:
use yggdryl::generic::{DataTypeId, IOMode, TimeUnit};

assert_eq!(DataTypeId::Int64.as_str(), "int64");
assert_eq!(TimeUnit::Millisecond.as_str(), "ms");
assert_eq!(IOMode::ReadOnly.as_str(), "readonly");

In [ ]:
use yggdryl::{EnumScalar, IOMode, Scalar};

let value = Scalar::from(IOMode::Append);
let member = value.as_enum().expect("an enum scalar");
assert_eq!(member, &EnumScalar::IOMode(IOMode::Append));
assert_eq!((member.kind(), member.as_str(), member.ordinal()), ("io_mode", "append", 1));

In [ ]:
use yggdryl::generic::Holder;
use yggdryl::io::{Buffer, IOBase};

// A value that could have been any handle. The calls do not change.
let mut handle = Holder::buffer(Buffer::new());
handle.write_all_bytes(b"AAPL,1\n")?;

assert_eq!(handle.read_all_bytes()?, b"AAPL,1\n");
assert_eq!(handle.kind(), yggdryl::IOKind::Memory);

## Holder: every storage handle

In [ ]:
use yggdryl::generic::Holder;

// Generic construction records the location without probing its role.
let directory = Holder::local(std::env::temp_dir())?;
assert!(matches!(directory, Holder::Path(_)));

let missing = Holder::local(std::env::temp_dir().join("yggdryl-generic-doc.bin"))?;
assert!(matches!(missing, Holder::Path(_)));

In [ ]:
use yggdryl::generic::Holder;
use yggdryl::io::IOBase;

let root = Holder::folder(std::env::temp_dir())?;
assert!(root.is_container());

// A child need not exist. Naming one yields a leaf handle, and nothing is created.
let leaf = root.child_by_path("yggdryl-generic-child.bin")?;
assert!(matches!(leaf, Holder::File(_)));
assert!(!leaf.is_container());
assert_eq!(leaf.size(), 0);

## Codec: a coding over a handle

In [ ]:
use yggdryl::generic::Coded;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::Url;

let named = Buffer::new().with_media_type(Url::from_str("file:///trades.csv.zst")?.media_type());
let mut handle = Coded::infer(named);
assert_eq!(handle.codec(), yggdryl::Codec::Zstd);

handle.write_all_bytes(b"symbol,price\nAAPL,1\nAAPL,2\n")?;
handle.flush()?;

// The coded handle reads plain bytes; the handle underneath holds the frame.
assert_eq!(handle.read_all_bytes()?, b"symbol,price\nAAPL,1\nAAPL,2\n");
assert_ne!(handle.handle().as_slice(), b"symbol,price\nAAPL,1\nAAPL,2\n");

In [ ]:
use yggdryl::generic::Coded;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::Level;

let mut handle = Coded::wrap(Buffer::new(), yggdryl::Codec::Gzip).with_level(Level::BEST);
handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;

// into_handle publishes the pending write, then gives back the compressed bytes.
let inner = handle.into_handle()?;
assert_eq!(yggdryl::gzip::load(inner.as_slice())?, b"symbol,price\nAAPL,1\n");

In [ ]:
use yggdryl::generic::Coded;
use yggdryl::io::Buffer;

let handle = Coded::wrap(Buffer::new(), yggdryl::Codec::Deflate);
assert_eq!(handle.codec(), yggdryl::Codec::Zlib);

## Media: a record encoding over a handle

In [ ]:
use yggdryl::generic::{Holder, Media};
use yggdryl::io::Buffer;
use yggdryl::Url;

fn named(name: &str) -> Result<Holder, Box<dyn std::error::Error>> {
    let url = Url::from_str(&format!("file:///{name}"))?;
    Ok(Holder::buffer(Buffer::new().with_media_type(url.media_type())))
}

assert!(matches!(Media::open(named("trades.arrows")?)?, Media::Ipc(_)));
assert!(matches!(Media::open(named("trades.parquet")?)?, Media::Parquet(_)));
assert!(matches!(Media::open(named("app.log")?)?, Media::Text(_)));

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::{Holder, Media};
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

let url = Url::from_str("file:///trades.arrows")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));
let mut media = Media::open(handle)?.with_field(schema.clone());
let options = media.record_options()?;

media.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
assert_eq!(media.read_arrow_reader(&options)?.count(), 1);
assert_eq!(media.read_arrow_field(&options)?, schema);

// A Media is also the bytes it encodes: an Arrow IPC stream opens with its
// continuation marker.
assert_eq!(media.read_range(0, 4)?, [0xFF, 0xFF, 0xFF, 0xFF]);

### Measured generic media redirection

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::{Holder, Media};
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![9]))],
)?;

let url = Url::from_str("file:///trades.arrows.gz")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));
let mut media = Media::open(handle)?.with_field(schema.clone());
let options = media.record_options()?;

media.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
assert_eq!(media.read_arrow_reader(&options)?.count(), 1);

// Still an Arrow IPC stream, now behind gzip framing.
assert_eq!(media.read_range(0, 2)?, [0x1F, 0x8B]);

In [ ]:
use yggdryl::generic::{Holder, Media};
use yggdryl::io::Buffer;
use yggdryl::Url;

let url = Url::from_str("file:///trades.csv")?;
let handle = Holder::buffer(Buffer::new().with_media_type(url.media_type()));

let message = Media::open(handle).unwrap_err().to_string();
assert!(message.contains("text/csv"), "{message}");

## RecordOptions: every encoding's settings

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::{DataType, MimeType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let options = RecordOptions::for_media_type(&Url::from_str("file:///trades.parquet")?.media_type())?
    .with_field(schema.clone())
    .with_batch_size(1024);

assert_eq!(options.mime_type(), MimeType::PARQUET);
assert_eq!(options.field(), Some(&schema));
assert_eq!(options.batch_size(), Some(1024));
assert_eq!(options.stable_hash(), options.clone().stable_hash());

In [ ]:
use arrow_array::RecordBatch;
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::{DataType, MimeType};

let declared = DataType::from_fields([
    DataType::Utf8.required_field("symbol"),
    DataType::Int64.required_field("price"),
])?
.required_field("row");

let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?
    .with_field(declared.clone())
    .with_select_by_names(["price"]);

// One call is the whole pipeline: the declared cast, then the selection.
// Passing a stored root as the second argument adds the completion layer.
let batch = RecordBatch::new_empty(declared.into_arrow_schema()?);
let cast = options.cast_arrow_batch(batch, None)?;
assert_eq!(cast.num_columns(), 1);

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::ipc::IpcOptions;
use yggdryl::MimeType;

let options: RecordOptions = IpcOptions::new()
    .with_root_name("trade")
    .with_safe(false)
    .with_commit_row_size(10_000)
    .into();

assert_eq!(options.mime_type(), MimeType::ARROW_STREAM);
assert_eq!(options.root_name(), "trade");
assert!(!options.safe());
assert_eq!(options.commit_row_size(), Some(10_000));

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::MimeType;

let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?;
assert!(options.field().is_none());

let message = options.require_field().unwrap_err().to_string();
assert!(message.contains("with_field"), "{message}");

## Scalar families

In [ ]:
use yggdryl::{I256, Scalar, TemporalFamily, TimeUnit, Timezone};

let date = Scalar::from_date(20_000, TimeUnit::Day, Timezone::NAIVE)?;
let time = Scalar::from_time(1, TimeUnit::Nanosecond, Timezone::NAIVE)?;
let duration = Scalar::from_duration(i64::from(i32::MAX) + 1, TimeUnit::Second, Timezone::NAIVE)?;
let decimal = Scalar::from_decimal(I256::from_i128(1_250), 2);

assert_eq!(date.as_date().unwrap().bit_width(), 32);
assert_eq!(time.as_time().unwrap().bit_width(), 64);
assert_eq!(duration.as_duration().unwrap().bit_width(), 64);
assert_eq!(time.as_temporal().unwrap().family(), TemporalFamily::Time);
assert_eq!(decimal.as_decimal(), Some((I256::from_i128(1_250), 2)));

## TypedScalar: one value and its datatype

In [ ]:
use yggdryl::generic::{Int64Scalar, TypedScalar};
use yggdryl::{DataType, Scalar};

let price = TypedScalar::from_parts(DataType::Int64, Scalar::from(7_i64))?;
assert_eq!(price.data_type(), &DataType::Int64);

// The same pairing, with the datatype fixed at compile time.
let typed: Int64Scalar = price.try_into_typed()?;
assert_eq!(typed.value(), &Scalar::I64(7));
assert!(Int64Scalar::new(Scalar::from("seven")).is_err());

In [ ]:
use yggdryl::Scalar;

let scalar = Scalar::from(42_i64).inferred_scalar_field()?;
let array = Scalar::from_sequence([Scalar::from(1_i64), Scalar::Null]);
let row = Scalar::from_record([("id", Scalar::from(1_i64))])?;
let rows = Scalar::from_sequence([row]);

assert_eq!(scalar.name(), "value");
assert_eq!(array.inferred_array_field()?.name(), "item");
assert_eq!(rows.inferred_struct_field()?.name(), "row");

## The WKB reader

In [ ]:
use yggdryl::generic::wkb::{self, Geometry};

// A little-endian XY point: order byte, type code 1, then x and y.
let mut point = vec![1, 1, 0, 0, 0];
point.extend(10.0_f64.to_le_bytes());
point.extend(20.0_f64.to_le_bytes());

let decoded = Geometry::from_slice(&point)?;
assert_eq!(decoded.clone().into_wkt(), "POINT (10 20)");
assert_eq!(decoded.type_id(), 1);
assert!(!decoded.is_empty());

// The free functions answer without materializing the geometry.
assert_eq!(wkb::into_wkt(&point)?, "POINT (10 20)");
assert_eq!(wkb::geometry_type_ids(&point)?, [1]);
let bounds = wkb::bounding_box(&point)?;
assert_eq!((bounds.xmin, bounds.xmax, bounds.ymin, bounds.ymax), (10.0, 10.0, 20.0, 20.0));

In [ ]:
use yggdryl::generic::wkb;

// Truncated input: the error names the byte position.
let error = wkb::bounding_box(&[1, 1, 0, 0, 0]).unwrap_err();
assert!(error.to_string().contains("byte 5"), "{error}");